In [ ]:
# Import necessary libraries (optimized imports)
import pandas as pd
import numpy as np
import os
import csv

# PyTorch imports
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import TensorDataset, DataLoader, RandomSampler

# Scikit-learn imports
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score

# TensorFlow Hub / BERT imports
import tensorflow_hub as hub
import tensorflow_text

# ----------------------
# Configuration & Paths
# ----------------------
DOMAIN = 'Sports'  # Fashion, Game, Health, Sports
BERT_PREPROCESS_URL = 'https://kaggle.com/models/tensorflow/bert/frameworks/TensorFlow2/variations/en-uncased-preprocess/versions/3'
BERT_ENCODER_URL = 'https://www.kaggle.com/models/tensorflow/bert/frameworks/TensorFlow2/variations/bert-en-uncased-l-10-h-768-a-12/versions/2'
EPOCHS = 100
BATCH_SIZE = 8
LEARNING_RATE = 1e-3
WEIGHT_DECAY = 1e-5
OUTPUT_DIR = f'../Results/Proposed/{DOMAIN}_results/'
INPUT_DIR = '../Input_Data/'

# Create output directory if it doesn't exist
os.makedirs(OUTPUT_DIR, exist_ok=True)

# ----------------------
# Device Configuration
# ----------------------
device = "cuda" if torch.cuda.is_available() else "mps" if torch.backends.mps.is_available() else "cpu"
print(f"Using device: {device}")

# ----------------------
# Load BERT Models
# ----------------------
bert_preprocess_model = hub.KerasLayer(BERT_PREPROCESS_URL)
bert_model = hub.KerasLayer(BERT_ENCODER_URL)

# ----------------------
# BERT Embedding Function
# ----------------------
def get_bert_embedding(text):
    """Generate BERT pooled embedding for input text"""
    preprocessed_text = bert_preprocess_model(text)
    outputs = bert_model(preprocessed_text)
    return np.array(outputs["pooled_output"])

# ----------------------
# Load Topic Data & Embeddings
# ----------------------
current_topics = pd.read_csv(os.path.join(INPUT_DIR, 'Topic_Deepseek.csv'))
h_dim = len(current_topics)

# Generate topic embeddings
topic_embeddings = []
topic_confidence = []
for idx in range(len(current_topics)):
    term = current_topics.iloc[idx]['Terms']
    confidence = current_topics.iloc[idx]['Confidence']
    topic_embeddings.append(get_bert_embedding([term])[0])
    topic_confidence.append(confidence)
topic_embeddings = np.array(topic_embeddings)
topic_emb_tensor = torch.tensor(topic_embeddings, dtype=torch.float32).to(device)

topic_confidence = np.array(topic_confidence)
topic_conf_tensor = torch.tensor(topic_confidence, dtype=torch.float32).to(device)

# ----------------------
# Load Post Data & Embeddings
# ----------------------
post_df = pd.read_csv(os.path.join(INPUT_DIR, f'{DOMAIN}_Input Data.csv'))
post_embeddings = np.genfromtxt(
    os.path.join(INPUT_DIR, f'{DOMAIN}_post_embedding/Final_Posts_Dataset_final.csv'),
    delimiter=',',
    dtype=np.float32
)

# Extract labels (fixed redundant loop)
all_labels = post_df['Change_Indicator'].values.tolist()
all_post_emb = post_embeddings
print(f"Post embeddings shape: {all_post_emb.shape}")

# ----------------------
# Neural Network Model
# ----------------------
class NeuralNetwork(nn.Module):
    def __init__(self, input_dim=1536):
        super().__init__()
        # Fixed: Define input dimension explicitly
        self.fc_layer = nn.Sequential(
            nn.Linear(input_dim, 1),
            nn.Sigmoid()
        )

    def forward(self, post_x, topic_emb, topic_conf):
        # Topic attention score
        relevance_score = torch.matmul(post_x, topic_emb.T)
        confidence_score = topic_conf
        afflation_score = nn.Softmax(dim=-1)(relevance_score * confidence_score)

        # Concatenate post and topic embeddings
        batch_size = post_x.shape[0]
        num_topics = topic_emb.shape[0]

        post_expanded = post_x.unsqueeze(1).expand(batch_size, num_topics, -1)
        topic_expanded = topic_emb.unsqueeze(0).expand(batch_size, -1, -1)
        concatenated = torch.cat([post_expanded, topic_expanded], dim=2)

        # Forward through fully connected layer
        output = self.fc_layer(concatenated)
        output = output.squeeze(-1)

        # Weighted sum
        weighted_output = torch.mul(afflation_score, output)
        final_score = torch.sum(weighted_output, dim=1)

        return final_score, afflation_score, output

# Initialize model, optimizer, loss
model = NeuralNetwork().to(device)
optimizer = optim.Adam(model.parameters(), lr=LEARNING_RATE, weight_decay=WEIGHT_DECAY)
criterion = nn.BCELoss()

# ----------------------
# Training & Testing Functions
# ----------------------
def train_one_epoch(train_loader, topic_emb, topic_conf):
    model.train()
    total_loss = 0.0
    for x_batch, y_batch in train_loader:
        optimizer.zero_grad()
        outputs, _, _ = model(x_batch, topic_emb, topic_conf)
        loss = criterion(outputs, y_batch)
        loss.backward()
        optimizer.step()
        total_loss += loss.item()
    return total_loss / len(train_loader)

def evaluate(test_loader, topic_emb, topic_conf):
    model.eval()
    all_preds = []
    with torch.no_grad():
        for x_batch, _ in test_loader:
            outputs, _, _ = model(x_batch, topic_emb, topic_conf)
            preds = np.round(outputs.cpu().numpy()).tolist()
            all_preds.extend(preds)
    return all_preds

# ----------------------
# K-Fold Evaluation
# ----------------------
acc_all, prc_all, rec_all, f1_all = [], [], [], []
pred_all = []

for fold in range(1, 11):
    fold_col = f'K-{fold}'
    x_train, x_test, y_train, y_test = [], [], [], []

    # Split train/test based on fold column
    for idx in range(len(all_post_emb)):
        if post_df.iloc[idx][fold_col] == 'training':
            x_train.append(all_post_emb[idx])
            y_train.append(all_labels[idx])
        else:
            x_test.append(all_post_emb[idx])
            y_test.append(all_labels[idx])

    # Convert to tensors and create DataLoaders
    train_dataset = TensorDataset(
        torch.tensor(x_train, dtype=torch.float32).to(device),
        torch.tensor(y_train, dtype=torch.float32).to(device)
    )
    test_dataset = TensorDataset(
        torch.tensor(x_test, dtype=torch.float32).to(device),
        torch.tensor(y_test, dtype=torch.float32).to(device)
    )

    train_loader = DataLoader(train_dataset, sampler=RandomSampler(train_dataset), batch_size=BATCH_SIZE)
    test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE)

    # Training loop (fixed infinite while loop)
    best_acc = 0.0
    best_prc = best_rec = best_f1 = 0.0

    for _ in range(EPOCHS):
        train_one_epoch(train_loader, topic_emb_tensor, topic_conf_tensor)

    y_pred = evaluate(test_loader, topic_emb_tensor, topic_conf_tensor)

    # Calculate metrics
    acc = accuracy_score(y_test, y_pred)
    prc = precision_score(y_test, y_pred, zero_division=0)
    rec = recall_score(y_test, y_pred, zero_division=0)
    f1 = f1_score(y_test, y_pred, zero_division=0)

    # Store results
    pred_all.extend(y_pred)
    acc_all.append(acc)
    prc_all.append(prc)
    rec_all.append(rec)
    f1_all.append(f1)

# ----------------------
# Save Final Results
# ----------------------
# Metrics
metrics_df = pd.DataFrame({
    'accuracy': acc_all,
    'precision': prc_all,
    'recall': rec_all,
    'f1': f1_all
})
metrics_df.to_csv(os.path.join(OUTPUT_DIR, 'metrics.csv'), index=False)

# Predictions
pred_df = pd.DataFrame({
    'predictions': pred_all,
    'ground truth': all_labels[:len(pred_all)]
})
pred_df.to_csv(os.path.join(OUTPUT_DIR, 'predictions.csv'), index=False)

print("Training and testing completed. Results saved to:", OUTPUT_DIR)